In [10]:
import os
os.chdir('./../')

In [11]:
import pandas as pd
import numpy as np

In [12]:
sngp_tang_tang =  pd.read_csv('csv/tang_sngp/sngp_tang_tang.csv')
sngp_tang_wong =  pd.read_csv('csv/tang_sngp/sngp_tang_wong.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'csv/tang_sngp/sngp_tang_tang.csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def indist_vs_outdist_rejection(in_dist_csv: list, out_dist_csv: list, nbins: int = 10, plot: bool = True):
    """
    Given lists of DataFrames (each containing column 'prediction_prob_score'),
    compute per-bin percentages for in-distribution vs out-of-distribution data
    and plot them against mean thresholds.
    """
    col = "prediction_prob_score"
    
    # Combine all DataFrames in each list into one
    in_df = pd.concat(in_dist_csv, ignore_index=True)
    out_df = pd.concat(out_dist_csv, ignore_index=True)

    # Extract and sanitize the score columns
    in_scores = pd.to_numeric(in_df[col], errors='coerce').dropna().clip(0.0, 1.0)
    out_scores = pd.to_numeric(out_df[col], errors='coerce').dropna().clip(0.0, 1.0)

    # Define bin edges and centers
    bins = np.linspace(0.0, 1.0, nbins + 1)
    bin_lb = bins[:-1]
    bin_ub = bins[1:]
    mean_thresholds = (bin_lb + bin_ub) / 2.0

    # Count number of samples in each bin
    in_counts, _ = np.histogram(in_scores, bins=bins)
    out_counts, _ = np.histogram(out_scores, bins=bins)

    # Convert counts to percentages
    percent_of_indist_data = (in_counts / len(in_scores)) * 100
    percent_of_outdist_data = (out_counts / len(out_scores)) * 100

    # Plot results
    if plot:
        plt.figure(figsize=(8, 5))
        plt.plot(mean_thresholds, percent_of_indist_data, 'o-', label='In-Distribution %')
        plt.plot(mean_thresholds, percent_of_outdist_data, 's-', label='Out-of-Distribution %')
        plt.xlabel('Mean Threshold (bin centers)')
        plt.ylabel('Percentage of Samples in Bin (%)')
        plt.title(f'In-Dist vs Out-Dist Distribution (nbins={nbins})')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.tight_layout()
        plt.show()

    # return {
    #     "mean_thresholds": mean_thresholds,
    #     "bin_lb": bin_lb,
    #     "bin_ub": bin_ub,
    #     "percent_of_indist_data": percent_of_indist_data,
    #     "percent_of_outdist_data": percent_of_outdist_data,
    # }


In [ ]:
indist_vs_outdist_rejection(
    in_dist_csv=[sngp_tang_tang],
    out_dist_csv=[sngp_tang_wong],
    nbins=20,)

In [60]:
sngp_tang_tang

,image_id,target,prediction,prediction_prob_score,true_bin_label,class_probs,fold
0,6774146b-822a-45a1-abf8-56fe624ac1a8,2,2,0.986941,1,"[0.00035138509701937437, 0.002183708595111966,...",train
1,47d1cd60-cb3f-4fd7-95fa-bf177873b2cf,2,2,0.996499,1,"[0.00016975609469227493, 0.0019775149412453175...",train
2,02941d5d-4937-4e2d-9f08-8a78bef3b151,1,1,0.875835,1,"[0.014661467634141445, 0.8758349418640137, 0.0...",train
3,f5c5de8a-c985-4ad5-8edb-ecc2d1c3ba4d,0,0,0.931405,1,"[0.9314054250717163, 0.027712738141417503, 0.0...",train
4,d0b547b5-3f2c-4685-9c4f-0f0738db07de,2,2,0.959161,1,"[0.0007490213611163199, 0.03854332119226456, 0...",train
...,...,...,...,...,...,...,...
80868,c88d3115-749c-4be8-9372-6a50194e619d,2,2,0.995075,1,"[0.00018761181854642928, 0.0017459833761677146...",test
80869,008790eb-784b-44d1-ba53-2d6c0285bc9e,2,2,0.993843,1,"[0.00021032066433690488, 0.001860017073340714,...",test
80870,51e840ee-0a75-4bc2-9b54-941ff9d7311d,2,2,0.990751,1,"[0.00032249133801087737, 0.002363377483561635,...",test
80871,76d33beb-8d79-41d4-882e-018603e92bc5,2,2,0.605622,1,"[0.03746736794710159, 0.049681805074214935, 0....",test


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List

def performance_plot(
    in_dist_dfs: List[pd.DataFrame],
    out_dist_dfs: List[pd.DataFrame],
    n_bins: int = 10,
    show_individual: bool = True,
    label_all: bool = True,
    in_score_col: str = "prediction_prob_score",
    in_target_col: str = "target",
    in_pred_col: str = "prediction",
    out_score_col: str = "prediction_prob_score",
    figsize: tuple = (9, 5)
):
    """
    Plot in-distribution accuracy vs. confidence and out-of-distribution rejection density vs. confidence.
    Uses fixed x-axis bins (not mean confidences), and no interpolation/extrapolation.
    """

    # Fixed bin edges and centers
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2.0

    def _bin_indices(scores: pd.Series) -> pd.Series:
        return pd.cut(scores.clip(0, 1), bins=edges, include_lowest=True, right=False, labels=False).fillna(n_bins-1).astype(int)

    fig, ax_acc = plt.subplots(figsize=figsize)
    ax_rej = ax_acc.twinx()

    # Colors
    color_acc = plt.cm.tab10(0)
    color_ood = plt.cm.tab10(1)

    # ---------------------------
    # In-distribution (accuracy)
    # ---------------------------
    in_all_idx, in_all_correct = [], []
    for df in in_dist_dfs:
        if not {in_score_col, in_target_col, in_pred_col}.issubset(df.columns):
            raise ValueError("Missing required columns in in_dist_dfs")

        s = df[in_score_col].astype(float)
        idx = _bin_indices(s)
        correct = (df[in_target_col].values == df[in_pred_col].values)
        acc = np.full(n_bins, np.nan)
        for b in range(n_bins):
            mask = (idx == b).values
            if mask.any():
                acc[b] = correct[mask].mean()

        if show_individual:
            ax_acc.plot(centers, acc, marker="o", linewidth=1, alpha=0.3, color=color_acc)

        in_all_idx.append(idx.values)
        in_all_correct.append(correct)

    if in_all_idx:
        idx_all = np.concatenate(in_all_idx)
        correct_all = np.concatenate(in_all_correct)
        acc_all = np.full(n_bins, np.nan)
        for b in range(n_bins):
            mask = (idx_all == b)
            if mask.any():
                acc_all[b] = correct_all[mask].mean()
        ax_acc.plot(
            centers, acc_all, linewidth=2.5, marker="o",
            label="In-dist accuracy" if label_all else None, color=color_acc
        )

    # ------------------------------------------
    # Out-of-distribution (rejection density)
    # ------------------------------------------
    out_all_idx = []
    for df in out_dist_dfs:
        if out_score_col not in df.columns:
            raise ValueError("Missing required column in out_dist_dfs")

        s = df[out_score_col].astype(float)
        idx = _bin_indices(s)
        counts = np.bincount(idx, minlength=n_bins).astype(float)
        prop = counts / counts.sum() if counts.sum() > 0 else np.zeros_like(counts)

        if show_individual:
            ax_rej.plot(centers, prop, marker="o", linewidth=1, alpha=0.3, color=color_ood)

        out_all_idx.append(idx)

    if out_all_idx:
        idx_all = np.concatenate(out_all_idx)
        counts = np.bincount(idx_all, minlength=n_bins).astype(float)
        prop_all = counts / counts.sum() if counts.sum() > 0 else np.zeros_like(counts)
        ax_rej.plot(
            centers, prop_all, linewidth=2.5, marker="o",
            label="OOD rejection density" if label_all else None, color=color_ood
        )

    # Ideal reference lines
    ax_acc.plot(centers, centers, linestyle="--", linewidth=1, color=color_acc,
                label="Ideal accuracy" if label_all else None)
    ax_rej.plot(centers, 1 - centers, linestyle="--", linewidth=1, color=color_ood,
                label="Ideal OOD density" if label_all else None)

    # Styling
    ax_acc.set_xlabel("Confidence bins")
    ax_acc.set_ylabel("Accuracy (in-distribution)", color=color_acc)
    ax_rej.set_ylabel("Density (out-of-distribution)", color=color_ood)
    ax_acc.set_xlim(0, 1)
    ax_acc.set_ylim(0, 1)
    ax_rej.set_ylim(0, 1)
    ax_acc.set_yticks(np.linspace(0, 1, 11))  # 10 evenly spaced y-ticks
    ax_rej.set_yticks(np.linspace(0, 1, 11))
    ax_acc.set_xticks(np.linspace(0, 1, n_bins + 1))
    ax_acc.grid(True, alpha=0.3)

    if label_all:
        lines_labels = []
        for ax in (ax_acc, ax_rej):
            handles, labels = ax.get_legend_handles_labels()
            lines_labels.extend(list(zip(handles, labels)))
        if lines_labels:
            handles, labels = zip(*[(h, l) for h, l in lines_labels if l])
            ax_acc.legend(handles, labels, loc="best", frameon=True)

    ax_acc.set_title("Accuracy vs. Confidence (In-Dist) & OOD Rejection Density")

    plt.tight_layout()
    return fig, (ax_acc, ax_rej)


In [ ]:
fig, (ax_acc, ax_rej) = performance_plot(
    in_dist_dfs=[sngp_tang_tang],
    out_dist_dfs=[sngp_tang_wong])

In [ ]:
sngp_tang_tang.head()